# Deploy Tellr to Databricks Apps

Run this notebook locally to deploy, update, or delete a Tellr instance on your Databricks workspace.

**Prerequisites:**
- A Databricks workspace with Apps enabled
- Permission to create a Lakebase (or a schema in an existing one)
- Authentication configured (CLI profile or environment variables)

## 1. Install Dependencies

In [7]:
%pip install databricks-tellr databricks-sdk ipywidgets

Looking in indexes: https://pypi-proxy.dev.databricks.com/simple
  Using cached databricks_tellr-0.2.7-py3-none-any.whl.metadata (723 bytes)
  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata (20 kB)
Using cached databricks_tellr-0.2.7-py3-none-any.whl (13 kB)
Using cached ipywidgets-8.1.8-py3-none-any.whl (139 kB)
Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl (914 kB)
Using cached widgetsnbextension-4.0.15-py3-none-any.whl (2.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [databricks-tellr]
Note: you may need to restart the kernel to use updated packages.


## 2. Configure Authentication

Pick **one** of the two options below and fill in your values.

In [7]:
# --- Option A: CLI profile from ~/.databrickscfg ---
PROFILE = "tellr"  # Change to your profile name

# --- Option B: Explicit credentials (uncomment and fill in) ---
# import os
# os.environ["DATABRICKS_HOST"] = "https://your-workspace.cloud.databricks.com"
# os.environ["DATABRICKS_TOKEN"] = "dapi..."
# PROFILE = None

## 3. Set Deployment Parameters

In [9]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import os

# Clear any cached env vars that might interfere
for var in ["DATABRICKS_HOST", "DATABRICKS_TOKEN", "DATABRICKS_AUTH_TYPE"]:
    os.environ.pop(var, None)

# Environment configurations
ENV_CONFIGS = {
    "dev": {
        "profile": "DEFAULT",
        "workspace_user": "huong.vu@databricks.com",
        "app_prefix": "tellr",  # Results in: tellr-dev
        "lakebase_prefix": "tellr-db",  # Results in: tellr-db-dev
    },
    "stage": {
        "profile": "tellr",
        "workspace_user": "huong.vu@databricks.com",
        "app_prefix": "db-tellr",  # Results in: db-tellr-stage
        "lakebase_prefix": "db-tellr-db",  # Results in: db-tellr-db-stage
    },
    # Uncomment when ready for production
    "prod": {
        "profile": "DEFAULT",
        "workspace_user": "huong.vu@databricks.com",
        "app_prefix": "db-tellr", #results in: db-tellr-prod
        "lakebase_prefix": "db-tellr",
    },
}

# Global variables (updated by dropdown)
PROFILE = None
LAKEBASE_NAME = None
SCHEMA_NAME = None
APP_NAME = None
APP_FILE_WORKSPACE_PATH = None

# Output area for displaying current config
config_output = widgets.Output()

def update_config(stage: str):
    """Update global config variables based on selected stage."""
    global PROFILE, LAKEBASE_NAME, SCHEMA_NAME, APP_NAME, APP_FILE_WORKSPACE_PATH
    
    env = ENV_CONFIGS[stage]
    PROFILE = env["profile"]
    app_prefix = env.get("app_prefix", "tellr")
    lakebase_prefix = env.get("lakebase_prefix", "db-tellr")
    
    if stage !='prod':
        LAKEBASE_NAME = f"{lakebase_prefix}-{stage}"
    else:
        LAKEBASE_NAME = f"{lakebase_prefix}"
    SCHEMA_NAME = f"tellr_app_data_{stage}"
    APP_NAME = f"{app_prefix}-{stage}"
    # Correct path format: /Workspace/Users/{user}/.apps/{stage}/tellr
    APP_FILE_WORKSPACE_PATH = f"/Workspace/Users/{env['workspace_user']}/.apps/{stage}/tellr"
    
    with config_output:
        clear_output(wait=True)
        print(f"✓ Environment: {stage.upper()}")
        print(f"  Profile:    {PROFILE}")
        print(f"  Lakebase:   {LAKEBASE_NAME}")
        print(f"  Schema:     {SCHEMA_NAME}")
        print(f"  App Name:   {APP_NAME}")
        print(f"  Workspace:  {APP_FILE_WORKSPACE_PATH}")

def on_env_change(change):
    if change["type"] == "change" and change["name"] == "value":
        update_config(change["new"])

# Build dropdown options dynamically from ENV_CONFIGS
ENV_LABELS = {
    "dev": "🔧 Development (dev)",
    "stage": "🧪 Staging (stage)",
    "prod": "🚀 Production (prod)",
}
dropdown_options = [(ENV_LABELS.get(k, k), k) for k in ENV_CONFIGS.keys()]

# Create dropdown
env_dropdown = widgets.Dropdown(
    options=dropdown_options,
    value=list(ENV_CONFIGS.keys())[0],  # Default to first available env
    description="Environment:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="300px"),
)
env_dropdown.observe(on_env_change)

# Display widgets
display(widgets.VBox([
    widgets.HTML("<h4>Select Deployment Environment</h4>"),
    env_dropdown,
    config_output,
]))

# Initialize with default selection
update_config(list(ENV_CONFIGS.keys())[0])

## 4. Deploy for the First Time

In [3]:
import os
for var in ["DATABRICKS_HOST", "DATABRICKS_TOKEN", "DATABRICKS_AUTH_TYPE"]:
    print(os.environ.get(var))

None
None
None


In [4]:
import databricks_tellr as tellr

result = tellr.create(
    lakebase_name=LAKEBASE_NAME,
    schema_name=SCHEMA_NAME,
    app_name=APP_NAME,
    app_file_workspace_path=APP_FILE_WORKSPACE_PATH,
    profile=PROFILE,
)

print(f"App URL: {result['url']}")

Deploying Tellr to Databricks Apps...
   App name: db-tellr-stage
   Workspace path: /Workspace/Users/huong.vu@databricks.com/.apps/stage/tellr
   Lakebase: db-tellr-db-stage (capacity: CU_1)
   Schema: tellr_app_data_stage

Setting up Lakebase database...
   Lakebase: db-tellr-db-stage (exists, type=autoscaling)

Preparing deployment files...
   Generated requirements.txt
   Generated app.yaml (with encryption key)
Uploading to: /Workspace/Users/huong.vu@databricks.com/.apps/stage/tellr
   Files uploaded

Creating Databricks App: db-tellr-stage
   App registered

Configuring SP role on autoscaling project...
   Creating SP Postgres role via API: f1927d2d-4d8c-4363-b238-359ef5bbd00d
   SP role configured with LAKEBASE_OAUTH_V1 auth
Setting up database schema...
   Schema 'tellr_app_data_stage' configured

Deploying app...
   App deployed
   URL: https://db-tellr-stage-7474645249186749.aws.databricksapps.com

Deployment complete!
App URL: https://db-tellr-stage-7474645249186749.aws.data

---
## 5. Dev Deploy (Source Push)

Push local Python source files directly to the running Databricks App — no wheel build, no PyPI.

- **Dev Setup** (run once): uploads source + deps + dev app.yaml, triggers first deploy
- **Dev Push** (run on each code change): uploads only `src/`, triggers redeploy

### [OPTIONAL] Run database migrations

Execute this before redeploying if the schema change requires an ALTER (e.g. JSON → JSONB).

In [7]:
import psycopg2
from databricks.sdk import WorkspaceClient

ws = WorkspaceClient(profile=PROFILE)

# Resolve connection info (same logic as deploy.py)
project_name = f"projects/{LAKEBASE_NAME}"
endpoints = list(ws.postgres.list_endpoints(
    parent=f"{project_name}/branches/production"
))
endpoint = ws.postgres.get_endpoint(name=endpoints[0].name)
host = endpoint.status.hosts.host
cred = ws.postgres.generate_database_credential(endpoint=endpoints[0].name)
user = ws.current_user.me().user_name

conn = psycopg2.connect(
    host=host, port=5432, user=user, password=cred.token,
    dbname="databricks_postgres", sslmode="require",
)
conn.autocommit = True

with conn.cursor() as cur:
    # Option A: ALTER a single column (requires table ownership)
    # cur.execute(f'ALTER TABLE {SCHEMA_NAME}.image_assets ALTER COLUMN tags TYPE JSONB USING tags::JSONB;')

    # Option B: Reset schema — drop everything and let init_database() recreate
    # You own the schema so CASCADE works even on SP-owned tables.
    cur.execute(f'DROP SCHEMA IF EXISTS "{SCHEMA_NAME}" CASCADE')
    cur.execute(f'CREATE SCHEMA "{SCHEMA_NAME}"')

    # Re-grant permissions to the app's service principal
    cur.execute(f"""
        DO $$
        DECLARE r RECORD;
        BEGIN
            FOR r IN SELECT rolname FROM pg_roles WHERE rolname LIKE '%-%-%-%-%' LIMIT 1
            LOOP
                EXECUTE format('GRANT USAGE, CREATE ON SCHEMA %I TO %I', '{SCHEMA_NAME}', r.rolname);
                EXECUTE format('GRANT SELECT, INSERT, UPDATE, DELETE ON ALL TABLES IN SCHEMA %I TO %I', '{SCHEMA_NAME}', r.rolname);
                EXECUTE format('GRANT USAGE, SELECT ON ALL SEQUENCES IN SCHEMA %I TO %I', '{SCHEMA_NAME}', r.rolname);
                EXECUTE format('ALTER DEFAULT PRIVILEGES IN SCHEMA %I GRANT SELECT, INSERT, UPDATE, DELETE ON TABLES TO %I', '{SCHEMA_NAME}', r.rolname);
                EXECUTE format('ALTER DEFAULT PRIVILEGES IN SCHEMA %I GRANT USAGE, SELECT ON SEQUENCES TO %I', '{SCHEMA_NAME}', r.rolname);
                RAISE NOTICE 'Granted permissions to %', r.rolname;
            END LOOP;
        END $$;
    """)
    print(f"Schema '{SCHEMA_NAME}' reset. Tables will be recreated on next deploy.")

conn.close()

Schema 'tellr_app_data_stage' reset. Tables will be recreated on next deploy.


### Helper functions

In [5]:
import io
import subprocess
import tomllib
from pathlib import Path

import yaml
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.apps import AppDeployment
from databricks.sdk.service.workspace import ImportFormat

REPO_ROOT = Path.cwd().parent  # assumes notebook is in <repo>/notebooks/

SKIP_PATTERNS = {"__pycache__", ".pyc", ".DS_Store"}


def upload_dir(ws, local_dir: Path, workspace_parent: str) -> int:
    """Recursively upload a local directory to {workspace_parent}/{dir.name}/..."""
    local_dir = Path(local_dir)
    dir_name = local_dir.name
    created_dirs: set[str] = set()
    count = 0
    for file_path in local_dir.rglob("*"):
        if not file_path.is_file():
            continue
        if any(p in file_path.parts for p in SKIP_PATTERNS) or file_path.suffix == ".pyc":
            continue
        rel = file_path.relative_to(local_dir)
        ws_file = f"{workspace_parent}/{dir_name}/{rel}"
        ws_dir = ws_file.rsplit("/", 1)[0]
        if ws_dir not in created_dirs:
            try:
                ws.workspace.mkdirs(ws_dir)
            except Exception:
                pass
            created_dirs.add(ws_dir)
        with open(file_path, "rb") as f:
            ws.workspace.upload(ws_file, f, format=ImportFormat.AUTO, overwrite=True)
        count += 1
    return count


def upload_dir_to(ws, local_dir: Path, workspace_target: str) -> int:
    """Upload contents of local_dir into workspace_target (custom target path)."""
    local_dir = Path(local_dir)
    created_dirs: set[str] = set()
    count = 0
    for file_path in local_dir.rglob("*"):
        if not file_path.is_file():
            continue
        if any(p in file_path.parts for p in SKIP_PATTERNS):
            continue
        rel = file_path.relative_to(local_dir)
        ws_file = f"{workspace_target}/{rel}"
        ws_dir = ws_file.rsplit("/", 1)[0]
        if ws_dir not in created_dirs:
            try:
                ws.workspace.mkdirs(ws_dir)
            except Exception:
                pass
            created_dirs.add(ws_dir)
        with open(file_path, "rb") as f:
            ws.workspace.upload(ws_file, f, format=ImportFormat.AUTO, overwrite=True)
        count += 1
    return count


def upload_bytes(ws, content: bytes, workspace_path: str):
    """Upload raw bytes to a workspace file path."""
    ws.workspace.upload(workspace_path, io.BytesIO(content), format=ImportFormat.AUTO, overwrite=True)



### Dev Setup (run once, or when dependencies change)

In [12]:
ws = WorkspaceClient(profile=PROFILE)
try:
    ws.workspace.mkdirs(APP_FILE_WORKSPACE_PATH)
except Exception:
    pass

# 1. Generate deps-only requirements.txt from pyproject.toml
pyproject_path = REPO_ROOT / "packages" / "databricks-tellr-app" / "pyproject.toml"
with open(pyproject_path, "rb") as f:
    deps = tomllib.load(f)["project"]["dependencies"]

req_content = "# Dev mode — dependencies only, no app wheel\n" + "\n".join(deps) + "\n"
upload_bytes(ws, req_content.encode(), f"{APP_FILE_WORKSPACE_PATH}/requirements.txt")
print(f"Uploaded requirements.txt ({len(deps)} dependencies)")

# 2. Download existing app.yaml, swap command for dev-friendly version
resp = ws.workspace.download(f"{APP_FILE_WORKSPACE_PATH}/app.yaml")
raw = resp.read() if hasattr(resp, "read") else resp
app_yaml = yaml.safe_load(raw.decode("utf-8") if isinstance(raw, bytes) else str(raw))

app_yaml["command"] = [
    "sh", "-c",
    "pip install -r requirements.txt && "
    'python -c "from databricks_tellr_app.run import init_database; init_database()" && '
    "python -m databricks_tellr_app.run",
]

upload_bytes(
    ws,
    yaml.dump(app_yaml, default_flow_style=False, sort_keys=False).encode(),
    f"{APP_FILE_WORKSPACE_PATH}/app.yaml",
)
print("Uploaded app.yaml (dev mode)")

# 3. Upload source code
n = upload_dir(ws, REPO_ROOT / "src", APP_FILE_WORKSPACE_PATH)
print(f"Uploaded src/ ({n} files)")

n = upload_dir(
    ws,
    REPO_ROOT / "packages" / "databricks-tellr-app" / "databricks_tellr_app",
    APP_FILE_WORKSPACE_PATH,
)
print(f"Uploaded databricks_tellr_app/ ({n} files)")

if (REPO_ROOT / "config").exists():
    n = upload_dir(ws, REPO_ROOT / "config", APP_FILE_WORKSPACE_PATH)
    print(f"Uploaded config/ ({n} files)")

# 4. Build frontend and upload as package assets
#    The app resolves frontend via: importlib.resources.files("databricks_tellr_app") / "_assets" / "frontend"
#    So we upload frontend/dist/ into databricks_tellr_app/_assets/frontend/ in the workspace.
frontend_dir = REPO_ROOT / "frontend"
frontend_dist = frontend_dir / "dist"

if frontend_dir.exists():
    if not frontend_dist.exists():
        print("Building frontend (npm install + build)...")
        subprocess.run(["npm", "install"], cwd=frontend_dir, check=True, capture_output=True)
        subprocess.run(["npm", "run", "build"], cwd=frontend_dir, check=True, capture_output=True)

    assets_target = f"{APP_FILE_WORKSPACE_PATH}/databricks_tellr_app/_assets/frontend"
    n = upload_dir_to(ws, frontend_dist, assets_target)
    print(f"Uploaded frontend assets ({n} files)")
else:
    print("Warning: frontend/ not found — app will run without UI")

# 5. Deploy
print("Deploying...")
deployment = AppDeployment(source_code_path=APP_FILE_WORKSPACE_PATH)
result = ws.apps.deploy_and_wait(app_name=APP_NAME, app_deployment=deployment)
print(f"Deployment complete: {result.deployment_id}")

app = ws.apps.get(name=APP_NAME)
if app.url:
    print(f"App URL: {app.url}")

Uploaded requirements.txt (35 dependencies)
Uploaded app.yaml (dev mode)
Uploaded src/ (122 files)
Uploaded databricks_tellr_app/ (2 files)
Uploaded config/ (4 files)
Uploaded frontend assets (7 files)
Deploying...
Deployment complete: 01f13ce060cd1d37bfe05d99a5ffb267
App URL: https://db-tellr-stage-7474645249186749.aws.databricksapps.com


### Dev Push (fast iteration)

Run this cell after changing Python code in `src/`. Uploads source files and restarts the app — no build step.

In [10]:
import subprocess
# import psycopg2
from databricks.sdk import WorkspaceClient
ws = WorkspaceClient(profile=PROFILE)


n = upload_dir(ws, REPO_ROOT / "src", APP_FILE_WORKSPACE_PATH)
print(f"Uploaded src/ ({n} files)")

# Rebuild and upload frontend (needed when frontend code changes)
frontend_dir = REPO_ROOT / "frontend"
frontend_dist = frontend_dir / "dist"
if frontend_dir.exists():
    print("Building frontend...")
    subprocess.run(["npm", "run", "build"], cwd=frontend_dir, check=True, capture_output=True)
    assets_target = f"{APP_FILE_WORKSPACE_PATH}/databricks_tellr_app/_assets/frontend"
    n_fe = upload_dir_to(ws, frontend_dist, assets_target)
    print(f"Uploaded frontend assets ({n_fe} files)")

print("Deploying...")
deployment = AppDeployment(source_code_path=APP_FILE_WORKSPACE_PATH)
result = ws.apps.deploy_and_wait(app_name=APP_NAME, app_deployment=deployment)
print(f"Deployment complete: {result.deployment_id}")

app = ws.apps.get(name=APP_NAME)
if app.url:
    print(f"App URL: {app.url}")

Uploaded src/ (130 files)
Building frontend...
Uploaded frontend assets (9 files)
Deploying...
Deployment complete: 01f14e17e1c0158f97dae1d6bb51a6eb
App URL: https://db-tellr-stage-7474645249186749.aws.databricksapps.com


---
## 6. Delete the App

Run this cell to tear down the deployment. Set `reset_database=True` to also drop the database schema.

In [ ]:
result = tellr.delete(
    app_name=APP_NAME,
    lakebase_name=LAKEBASE_NAME,
    schema_name=SCHEMA_NAME,
    reset_database=True,
    profile=PROFILE,
)

print(result)